# [실습] 전처리 파이프라인과 데이터 누수 — 점수를 믿어도 되는가

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# Chapter 4 실습. 이 점수를 믿어도 되는가?

**필수 머신러닝 · Chapter 4**

| Part | 주제 |
| --- | --- |
| Part 1 | 너무 좋은 점수 의심하기 |
| Part 2 | 전처리를 올바른 순서로 |
| Part 3 | 점수를 믿는 법: 교차검증과 시간 순 분할 |
| Part 4 | 보고할 숫자 정하기 |

## 분석 의뢰서

> **의뢰 부서**: 은행 마케팅팀 (Chapter 3의 후속 의뢰)
> **목적**: 지난번 모델을 다음 캠페인에 실제로 쓰려 한다. 그런데 팀원 한 명이 통화 시간 변수를 추가했더니 점수가 훨씬 좋아졌다고 한다.
> **제약**: 모델이 약속한 가입 수가 실제와 크게 다르면 캠페인 예산 승인을 받을 수 없다.
> **질문**: 어느 모델의 점수를 믿고 보고해야 하는가? 그 점수가 실제 캠페인에서도 나온다고 어떻게 확신하는가?

## 이 실습의 질문

> **통화 시간을 뺀 모델과 넣은 모델, 무작위 분할과 시간 순 분할 — 마케팅팀에 "다음 캠페인에서 기대할 가입 수"로 보고할 숫자는 어느 것인가? 그리고 그 숫자를 믿을 수 있다고 어떻게 설득할 것인가?**

이 실습에는 새 모델이 나오지 않습니다. Chapter 3의 모델을 그대로 쓰면서, 그 점수를 **의심하고, 고쳐 만들고, 다시 재는** 연습을 합니다.

## 이 노트북을 쓰는 법

| 표시 | 뜻 |
| --- | --- |
| **[직접 작성]** | 빈 코드 셀을 직접 채웁니다. 강의 교안의 코드를 보지 않고 먼저 써 보세요 |
| **[판단]** | 코드가 아니라 생각을 적는 칸입니다. 셀을 더블클릭해 `나의 답:` 뒤에 적으세요 |
| **[제공]** | 미리 채워 둔 코드입니다. 읽고 실행하세요 |
| **힌트** | 막혔을 때만 펼치세요 |
| **먼저 예상하기** | 결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다 |

## 준비

Chapter 3과 같은 `bank-full.csv`(통화 45,211건)를 이 노트북과 같은 폴더에 두세요. 이번에는 Chapter 3에서 뺐던 통화 시간(`duration`) 열을 **지우지 않고** 둡니다.

In [1]:
# [제공] 이 실습에서 쓰는 라이브러리
import platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# 그래프의 한글 글꼴 (환경에 맞는 글꼴이 없으면 글자가 깨질 수 있습니다)
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

In [2]:
# [환경] 그래프 한글 글꼴 등록 (같은 폴더의 fonts/NanumGothic-Regular.ttf)
import os
import matplotlib.font_manager as fm
for _p in ["fonts/NanumGothic-Regular.ttf", "../fonts/NanumGothic-Regular.ttf", "NanumGothic-Regular.ttf"]:
    if os.path.exists(_p):
        fm.fontManager.addfont(_p)
        plt.rcParams["font.family"] = fm.FontProperties(fname=_p).get_name()
        break
plt.rcParams["axes.unicode_minus"] = False

---
# Part 1. 너무 좋은 점수 의심하기

## 1-1. 준비: Chapter 3의 데이터 다시 만들기

**[제공]** Chapter 3에서 한 준비를 한 번에 합니다. 데이터를 불러와 `target`을 만들고, 같은 방법(80 : 20, `random_state=42`, `stratify`)으로 나눕니다. 이번에는 `duration` 열을 지우지 않습니다.

| 이름 | 뜻 |
| --- | --- |
| `num_cols` | 숫자 피처 4개: `age`, `balance`, `campaign`, `previous` |
| `cat_cols` | 범주 피처 8개: `poutcome`, `housing`, `loan`, `contact`, `month`, `job`, `marital`, `education` |
| `features` | 두 목록을 합친 Chapter 3 모델의 피처 12개 |

In [3]:
# [제공] 데이터 준비 (Chapter 3과 같음, duration은 남겨 둠)
df = pd.read_csv("bank-full.csv", sep=";")
df["target"] = (df["y"] == "yes").astype(int)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["target"])

num_cols = ["age", "balance", "campaign", "previous"]
cat_cols = ["poutcome", "housing", "loan", "contact", "month", "job", "marital", "education"]
features = num_cols + cat_cols
y_train, y_test = train_df["target"], test_df["target"]
print(len(train_df), len(test_df), "| 평가용 가입률", round(y_test.mean(), 3))

36168 9043 | 평가용 가입률 0.117


### 실습 도구 (실행만 하세요)

| 도구 | 하는 일 |
| --- | --- |
| `record(이름, 정답, 확률, 분할)` | 점수를 결과표에 남긴다: ROC AUC, PR AUC, 평가 데이터의 가입률, 상위 1,000명 중 가입 수, 무작위로 1,000명을 골랐을 때의 기대 가입 수, 배수 |
| `show_log()` | 결과표를 보여 준다 |

In [4]:
# [제공] 실습 도구
results = []

def record(name, y_true, prob, split="무작위"):
    y_true = np.asarray(y_true); prob = np.asarray(prob)
    top = y_true[np.argsort(-prob, kind="stable")][:1000].sum()
    base = y_true.mean()
    row = {"실험": name, "분할": split,
           "ROC AUC": round(roc_auc_score(y_true, prob), 3),
           "PR AUC": round(average_precision_score(y_true, prob), 3),
           "평가 가입률": round(base, 3),
           "상위 1,000명 가입": int(top),
           "무작위 1,000명 기대": round(1000 * base),
           "배수": round(top / (1000 * base), 1)}
    results.append(row)
    print(f"기록: [{name} / {split}] ROC AUC {row['ROC AUC']:.3f} | 상위 1,000명 가입 {row['상위 1,000명 가입']}명 (무작위의 {row['배수']}배)")

def show_log():
    return pd.DataFrame(results)

## 1-2. Chapter 3의 모델 다시 만들기

**[직접 작성]** Chapter 3과 같은 방법으로 로지스틱 회귀를 만들고, 평가용 고객의 가입 확률을 `prob_base`에 저장한 뒤 `record("Chapter 3 모델", y_test, prob_base)`로 기록하세요.

1. `pd.get_dummies(df[features])`로 원핫 인코딩 → 학습용·평가용 행으로 나누기 (`.loc[train_df.index]`)
2. `StandardScaler`를 **학습용으로** `fit`하고, 학습용·평가용을 `transform`
3. `LogisticRegression(max_iter=3000)`으로 학습하고 `predict_proba(...)[:, 1]`

<details><summary>힌트</summary>

```python
X = pd.get_dummies(df[features]).astype(float)
X_tr, X_te = X.loc[train_df.index], X.loc[test_df.index]
scaler = StandardScaler().fit(X_tr)
model = LogisticRegression(max_iter=3000).fit(scaler.transform(X_tr), y_train)
prob_base = model.predict_proba(scaler.transform(X_te))[:, 1]
```
</details>

In [5]:
X = pd.get_dummies(df[features]).astype(float)
X_tr, X_te = X.loc[train_df.index], X.loc[test_df.index]

scaler = StandardScaler().fit(X_tr)
model = LogisticRegression(max_iter=3000).fit(scaler.transform(X_tr), y_train)
prob_base = model.predict_proba(scaler.transform(X_te))[:, 1]
record("Chapter 3 모델", y_test, prob_base)

기록: [Chapter 3 모델 / 무작위] ROC AUC 0.773 | 상위 1,000명 가입 464명 (무작위의 4.0배)


결과가 Chapter 3과 같은지(ROC AUC 0.773 안팎) 확인하세요.

## 1-3. 통화 시간을 넣어 보면

### 먼저 예상하기

결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다. Chapter 3 실습에서 피처를 더하고 설정을 바꿨을 때 ROC AUC가 얼마나 움직였는지 떠올려 보세요.

> **[판단]** 예상: 통화 시간(`duration`, 초)을 피처에 하나 더하면 ROC AUC가 0.773에서 얼마가 될까요?
>
> 나의 답:

예상: 통화 시간을 더하면 ROC AUC가 0.773에서 **0.90 안팎**으로 크게 오를 것 같습니다(그래서 팀원이 놀랐을 것입니다). 다만 그 상승분은 모델이 더 똑똑해져서가 아니라, 전화가 끝난 뒤에만 알 수 있는 정보를 미리 본 결과일 것이라고 의심합니다.


**[직접 작성]** 1-2의 코드에서 피처 목록을 `features + ["duration"]`으로 바꿔 같은 모델을 만들고, 확률을 `prob_dur`에 저장한 뒤 `record("통화 시간 추가", y_test, prob_dur)`로 기록하세요.

In [6]:
feat_dur = features + ["duration"]
X_dur = pd.get_dummies(df[feat_dur]).astype(float)
X_tr_d, X_te_d = X_dur.loc[train_df.index], X_dur.loc[test_df.index]

scaler_d = StandardScaler().fit(X_tr_d)
model_d = LogisticRegression(max_iter=3000).fit(scaler_d.transform(X_tr_d), y_train)
prob_dur = model_d.predict_proba(scaler_d.transform(X_te_d))[:, 1]
record("통화 시간 추가", y_test, prob_dur)

기록: [통화 시간 추가 / 무작위] ROC AUC 0.906 | 상위 1,000명 가입 578명 (무작위의 4.9배)


> **[판단]** ROC AUC가 얼마나 올랐나요? 예상과 비교해 보세요. Chapter 3 실습에서 다른 피처를 더했을 때의 변화와 견주면, 이 변화는 어느 정도로 큰가요? 이 결과를 보고 먼저 기뻐해야 할까요, 먼저 의심해야 할까요?
>
> 나의 답:

ROC AUC는 **0.773 → 0.906**으로 **+0.133** 올랐고, 상위 1,000명 가입도 464명 → **578명**으로 114명 늘었습니다. Chapter 3에서 피처를 3개 더했을 때는 AUC가 오히려 -0.001, 상위 1,000명 가입이 +2명이었던 것과 비교하면, 이 변화는 **다른 종류의 변화**입니다. 변수 하나로 0.13이 오르는 일은 실무에서 거의 없습니다. 그래서 나는 먼저 기뻐하기보다 **의심**합니다. 이 변수가 예측 시점에 존재하는지, 타깃과 어떤 경로로 연결되는지를 확인해야 합니다.


## 1-4. 이 변수는 어디서 오는가

**[직접 작성]** **학습용 데이터**에서 다음 두 가지를 확인하세요.

1. 가입(`target` = 1)과 미가입(0) 고객의 통화 시간 하위 25%, 중앙값, 상위 25%
2. 통화 시간이 60초 미만인 통화와 600초(10분) 초과인 통화의 가입률과 통화 수

<details><summary>힌트</summary>

1번은 `train_df.groupby("target")["duration"].quantile([0.25, 0.5, 0.75]).unstack()`.
2번은 조건을 만들어 `train_df.loc[조건, "target"].agg(["mean", "size"])`.
</details>

In [7]:
print("[학습용] target별 통화 시간 분위수(초)")
print(train_df.groupby("target")["duration"].quantile([0.25, 0.5, 0.75]).unstack().round(0))

short = train_df["duration"] < 60
long_ = train_df["duration"] > 600
print()
print(f"60초 미만: 가입률 {train_df.loc[short, 'target'].mean():.3f} | 통화 {short.sum():,}건")
print(f"600초 초과: 가입률 {train_df.loc[long_, 'target'].mean():.3f} | 통화 {long_.sum():,}건")

[학습용] target별 통화 시간 분위수(초)
         0.25   0.50   0.75
target                     
0        95.0  164.0  278.0
1       245.0  426.0  734.0

60초 미만: 가입률 0.002 | 통화 3,727건
600초 초과: 가입률 0.486 | 통화 3,043건


> **[판단]** 통화 시간과 가입 사이의 관계는 어느 방향일까요? 통화가 길어서 가입한 것일까요, 가입할 고객이라서 통화가 길어진 것일까요? 마케팅팀이 "누구에게 전화할까"를 정하는 시점에 이 값을 알 수 있나요?
>
> 나의 답:

관계는 **명확한 양의 관계**입니다. 가입 고객의 통화 시간 중앙값은 426초, 미가입은 164초로 2.6배 차이이고, 60초 미만 통화는 3,727건 중 가입률이 **0.2%**, 600초 초과 통화는 3,043건 중 가입률이 **48.6%**입니다. 그런데 방향은 "통화가 길어서 가입했다"가 아니라 **"가입할 고객이라서 통화가 길어졌다"**에 가깝습니다. 상담원이 관심을 보이는 고객과 더 오래 이야기하고, 고객이 질문을 하면 통화가 길어지며, 가입 결정이 대화 끝에 이뤄지기 때문입니다. 마케팅팀이 "누구에게 전화할까"를 정하는 시점에는 이 값이 **존재하지 않습니다**. 전화를 걸어야만 생기는 값이므로, 예측에 쓰면 미래 정보를 쓰는 셈입니다.


## 1-5. 피처마다 예측 시점 점검하기

데이터 설명서에 적힌 각 열의 뜻입니다. 마케팅팀은 **전화를 걸기 전에** 누구에게 걸지 정해야 합니다.

| 열 | 설명서의 뜻 |
| --- | --- |
| `age`, `job`, `marital`, `education` | 고객 정보 |
| `balance`, `housing`, `loan` | 계좌 잔액, 주택 대출, 개인 대출 |
| `contact` | 연락 수단 |
| `month`, `day` | 마지막 통화의 월, 일 |
| `campaign` | 이번 캠페인에서 이 고객에게 건 통화 수 (**마지막 통화 포함**) |
| `pdays`, `previous`, `poutcome` | 이전 캠페인 이후 경과일, 이전 캠페인 통화 수, 이전 캠페인 결과 |
| `duration` | 마지막 통화의 시간 (초) |

> **[판단]** 위 열 중 예측 시점(전화를 걸기 전)에 **알 수 없는** 열과 **애매한** 열을 고르고, 각각 이유를 적으세요. (힌트: `campaign`의 "마지막 통화 포함"을 눈여겨보세요.)
>
> 나의 답:

**알 수 없는 열**
- `duration`: 통화가 끝나야 알 수 있는 값. 이 실습의 핵심 누수입니다.
- `campaign`: 설명서에 "이번 캠페인에서 건 통화 수(**마지막 통화 포함**)"라고 되어 있습니다. 오늘 3번째 전화를 걸 고객을 고르는 시점에는 그 3번째 통화가 아직 없으므로, 값에는 **미래의 그 통화**가 포함됩니다.
- `pdays`, `previous`, `poutcome`: "이전 캠페인"의 결과라 이번 캠페인 시작 시점에는 알 수 있습니다. 다만 이번 캠페인이 이미 진행 중이라면 마지막 통화 이후 갱신될 수 있어, **캠페인 시작 시점 기준**으로 고정해야 합니다.

**애매한 열**
- `month`, `day`: "마지막 통화의 월·일"입니다. 아직 전화하지 않은 고객에게는 존재하지 않는 값이고, 전화를 건 뒤에 채워집니다. 그런데 데이터에서는 전부 채워져 있으므로(이번 캠페인의 마지막 통화 기준) 예측 시점과 어긋납니다. 캠페인 일정을 정하는 값으로는 쓸 수 있지만, "이 고객의 마지막 통화 월"로 쓰면 위험합니다.
- `contact`: 연락 수단은 알 수 있지만, 데이터의 값이 "이번에 실제로 쓴 수단"이라면 같은 문제가 생깁니다.


---
# Part 2. 전처리를 올바른 순서로

Part 1의 누수는 **변수**에 숨어 있었습니다. 이 Part의 누수는 **코드의 순서**에 숨어 있습니다. 이제부터는 통화 시간을 쓰지 않고, Chapter 3 모델의 피처 12개(`features`)만 씁니다.

## 2-1. 스케일러의 기준을 전체 데이터로 잡으면

**[직접 작성]** 1-2와 같은 모델을 만들되, 스케일러를 **전체 데이터(학습용 + 평가용)**로 `fit`한 버전의 확률을 `prob_leak`에 저장하세요. 그다음 `prob_base`와 `prob_leak`의 ROC AUC를 소수점 넷째 자리까지 출력해 비교하세요. (이 실험은 결과표에 기록하지 않습니다.)

<details><summary>힌트</summary>

`StandardScaler().fit(X)`처럼 나누기 전의 `X` 전체로 `fit`하면 됩니다. 나머지는 1-2와 같습니다. 출력은 `print(round(roc_auc_score(y_test, prob_leak), 4))`.
</details>

In [8]:
scaler_all = StandardScaler().fit(X)          # 학습용 + 평가용 전체로 기준을 잡음 (누수)
model_leak = LogisticRegression(max_iter=3000).fit(scaler_all.transform(X_tr), y_train)
prob_leak = model_leak.predict_proba(scaler_all.transform(X_te))[:, 1]

print("평가용 ROC AUC")
print(f"  학습용으로만 fit  : {roc_auc_score(y_test, prob_base):.4f}")
print(f"  전체로 fit(누수)   : {roc_auc_score(y_test, prob_leak):.4f}")

평가용 ROC AUC
  학습용으로만 fit  : 0.7729
  전체로 fit(누수)   : 0.7729


> **[판단]** 두 점수는 얼마나 다른가요? 왜 이런 결과가 나왔을까요? 점수에 차이가 없다면, 그래도 "기준은 학습용에서만" 규칙을 지켜야 하는 이유는 무엇일까요?
>
> 나의 답:

두 점수는 **0.7729 vs 0.7729**로 차이가 없습니다. 이유는 스케일러가 각 열의 평균과 표준편차만 바꾸고, 평가용 데이터가 학습용과 같은 분포에서 나왔기 때문입니다. 표준화는 순위를 바꾸지 않고(선형 변환), 로지스틱 회귀는 스케일이 달라져도 같은 결정 경계를 찾습니다. 그런데도 "기준은 학습용에서만" 규칙을 지켜야 하는 이유는, 이번처럼 차이가 0인 것이 **결과론**이기 때문입니다. ① 평가용이 학습용과 다른 시기·다른 분포면 전체로 fit한 기준이 평가용 정보를 담아 점수가 낙관적으로 나옵니다. ② 여러 번 반복해 설정을 고치면 그 작은 차이가 누적됩니다. ③ 운영에서는 평가용이 아니라 **새 고객 한 명**이 들어오므로, 전체로 fit한 기준은 존재하지도 않습니다. 규칙은 "차이가 있을 때 지키는 것"이 아니라 **항상 지키는 절차**입니다.


## 2-2. 기준에 타깃이 들어가면: 타깃 인코딩

**타깃 인코딩**은 범주를 "그 범주의 가입률"로 바꾸는 인코딩입니다. 여기서는 통화한 날(월 + 일, 예: `may-12`)을 범주로 써서, 날마다 가입률을 계산해 피처 하나로 만듭니다.

**[제공]** 날짜 범주를 만듭니다.

In [9]:
# [제공] 통화한 날(월-일) 범주
day_key = df["month"] + "-" + df["day"].astype(str)
key_tr, key_te = day_key.loc[train_df.index], day_key.loc[test_df.index]
print("날짜 범주 수:", day_key.nunique(), "| 날짜당 통화 수 중앙값:", int(day_key.value_counts().median()))

날짜 범주 수: 318 | 날짜당 통화 수 중앙값: 29


**[직접 작성]** 날짜별 가입률을 두 가지 방법으로 계산해, 각각 **피처 하나짜리** 로지스틱 회귀를 만들고 평가용 ROC AUC를 비교하세요.

- **올바른 방법**: 가입률을 **학습용 데이터로만** 계산한다 (`train_df["target"].groupby(key_tr).mean()`)
- **누수가 있는 방법**: 가입률을 **전체 데이터로** 계산한다 (`df["target"].groupby(day_key).mean()`)

어느 방법이든 계산한 가입률을 학습용과 평가용 날짜에 `map`으로 붙여 피처로 씁니다. 학습용에 없던 날짜는 학습용 전체 가입률로 채웁니다.

<details><summary>힌트</summary>

```python
rate = train_df["target"].groupby(key_tr).mean()
f_tr = key_tr.map(rate).fillna(y_train.mean()).to_frame()
f_te = key_te.map(rate).fillna(y_train.mean()).to_frame()
m = LogisticRegression().fit(f_tr, y_train)
print(roc_auc_score(y_test, m.predict_proba(f_te)[:, 1]))
```
누수가 있는 방법은 `rate`만 전체 데이터로 바꿉니다.
</details>

In [10]:
prior = train_df["target"].mean()

# 올바른 방법: 학습용 데이터로만 날짜별 가입률을 계산
rate_tr = train_df["target"].groupby(key_tr).mean()
f_tr_ok = key_tr.map(rate_tr).fillna(prior)
f_te_ok = key_te.map(rate_tr).fillna(prior)

# 누수가 있는 방법: 전체 데이터로 계산
rate_all = df["target"].groupby(day_key).mean()
f_tr_leak = key_tr.map(rate_all)
f_te_leak = key_te.map(rate_all)

def single_feature_auc(f_tr, f_te):
    m = LogisticRegression().fit(np.asarray(f_tr).reshape(-1, 1), y_train)
    return roc_auc_score(y_test, m.predict_proba(np.asarray(f_te).reshape(-1, 1))[:, 1])

print(f"올바른 방법(학습용으로만): ROC AUC {single_feature_auc(f_tr_ok, f_te_ok):.4f}")
print(f"누수가 있는 방법(전체로 계산): ROC AUC {single_feature_auc(f_tr_leak, f_te_leak):.4f}")

올바른 방법(학습용으로만): ROC AUC 0.7287


누수가 있는 방법(전체로 계산): ROC AUC 0.7481


> **[판단]** 이번에는 두 방법의 점수가 다른가요? 2-1의 스케일러와 무엇이 다르기에 이번에는 차이가 났을까요? "그날 통화가 몇 건뿐인 날"이라면 누수가 더 커질까요, 작아질까요?
>
> 나의 답:

이번에는 두 방법의 점수가 **0.7287 vs 0.7481**로 **0.019** 차이가 났습니다. 스케일러와 다른 점은, 타깃 인코딩이 **타깃 값 자체를 피처에 넣는** 변환이라는 것입니다. 날짜별 가입률을 전체 데이터로 계산하면 평가용 고객의 정답이 그 날짜의 가입률에 섞여 들어가, 평가용 점수가 부풀려집니다. 스케일러는 정답을 쓰지 않으므로 누수의 크기가 0이었지만, 타깃 인코딩은 정답을 직접 쓰므로 누수가 점수에 드러납니다. "그날 통화가 몇 건뿐인 날"이라면 누수는 **더 커집니다**. 가입률이 1/1이나 0/1로 계산되어 그 고객의 정답이 거의 그대로 피처 값이 되기 때문입니다(이 데이터의 날짜당 통화 수 중앙값은 29건입니다).


## 2-3. `get_dummies`의 함정

운영에서는 하루치 고객이 조금씩 들어옵니다. **[제공]** 셀은 평가용 고객 중 20명만 뽑아 `get_dummies`로 바꾸고, 학습할 때의 열과 비교합니다.

In [11]:
# [제공] 새 고객 20명이 들어왔다면
new_customers = test_df.sample(20, random_state=1)
train_cols = pd.get_dummies(df[features]).columns
new_cols = pd.get_dummies(new_customers[features]).columns
print("학습할 때 열 수:", len(train_cols), "| 새 고객 20명의 열 수:", len(new_cols))
print("새 고객에게 없는 열 (예시 10개):", sorted(set(train_cols) - set(new_cols))[:10])

학습할 때 열 수: 46 | 새 고객 20명의 열 수: 38
새 고객에게 없는 열 (예시 10개): ['job_entrepreneur', 'job_self-employed', 'job_student', 'job_unknown', 'month_aug', 'month_jan', 'month_sep', 'poutcome_success']


> **[판단]** 열 수가 다르면 학습한 모델에 새 고객을 넣을 때 어떤 일이 생길까요? 지금까지 Chapter 1~3에서 이 문제를 겪지 않은 이유는 무엇일까요?
>
> 나의 답:

학습할 때는 46개 열이었는데 새 고객 20명은 **38개**입니다. `job_entrepreneur`, `month_aug`, `poutcome_success`처럼 그 20명에게 없는 범주가 열에서 빠졌기 때문입니다. 학습한 모델은 46개 열을 기대하므로, 38개짜리 입력을 넣으면 **열 수가 맞지 않아 오류**가 나거나(순서대로 맞춰 넣으면 조용히 엉뚱한 열에 값이 들어가) 예측이 망가집니다. Chapter 1~3에서 이 문제를 겪지 않은 이유는 `pd.get_dummies(df)`를 **전체 데이터에 한 번** 적용한 뒤 나눴기 때문입니다. 그때는 평가용에도 모든 범주가 있었습니다. 운영에서는 새 고객이 한 명씩 오므로, 학습 때의 열 목록을 고정하고 없는 범주는 0으로 채워야 합니다. `OneHotEncoder(handle_unknown="ignore")`가 이 일을 합니다.


## 2-4. 파이프라인으로 다시 만들기

**[직접 작성]** `ColumnTransformer`와 `Pipeline`으로 Chapter 3 모델을 다시 만드세요.

1. `preprocess`: 숫자 열(`num_cols`)에는 `StandardScaler()`, 범주 열(`cat_cols`)에는 `OneHotEncoder(handle_unknown="ignore")`
2. `pipe_model`: `Pipeline([("pre", preprocess), ("clf", LogisticRegression(max_iter=3000))])`
3. **학습용 데이터 그대로**(`train_df[features]`) `fit`하고, 평가용 확률을 `prob_pipe`에 저장해 `record("파이프라인", y_test, prob_pipe)`로 기록
4. 2-3의 `new_customers`에도 `predict_proba`가 동작하는지 확인

<details><summary>힌트</summary>

```python
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
pipe_model = Pipeline([("pre", preprocess), ("clf", LogisticRegression(max_iter=3000))])
pipe_model.fit(train_df[features], y_train)
```
</details>

In [12]:
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
pipe_model = Pipeline([("pre", preprocess), ("clf", LogisticRegression(max_iter=3000))])

pipe_model.fit(train_df[features], y_train)
prob_pipe = pipe_model.predict_proba(test_df[features])[:, 1]
record("파이프라인", y_test, prob_pipe)
print("전처리 뒤 피처 수:", len(pipe_model.named_steps["pre"].get_feature_names_out()))

기록: [파이프라인 / 무작위] ROC AUC 0.773 | 상위 1,000명 가입 465명 (무작위의 4.0배)
전처리 뒤 피처 수: 46


> **[판단]** 파이프라인의 점수는 1-2의 "손으로 순서 맞추기"와 같은가요? 점수가 같다면, 파이프라인으로 바꿔서 얻은 것은 무엇인가요? 2-1, 2-3과 연결해 적어 보세요.
>
> 나의 답:

파이프라인의 평가용 ROC AUC는 **0.773**(상위 1,000명 465명)으로, 1-2의 손으로 맞춘 순서(0.773, 464명)와 사실상 같습니다. 같다면 파이프라인으로 바꿔서 얻은 것은 **점수가 아니라 안전장치**입니다. ① 2-1처럼 스케일러를 전체로 fit하는 실수를 구조적으로 막습니다(전처리가 학습용에만 fit됩니다). ② 2-3처럼 새 고객의 열이 달라지는 문제를 `handle_unknown="ignore"`로 막습니다. ③ 교차검증·그리드서치에 모델을 통째로 넣을 수 있어, fold마다 전처리가 다시 학습됩니다. 즉 "사람이 순서를 지켜야 하는 코드"에서 "순서가 코드에 박힌 코드"로 바뀐 것입니다.


---
# Part 3. 점수를 믿는 법: 교차검증과 시간 순 분할

## 3-1. 5겹 교차검증

Chapter 1~3에서는 평가용 점수를 보며 설정(깊이, $\alpha$, 임계값)을 골랐습니다. 이제는 설정을 고르는 데 **교차검증**을 쓰고, 평가용은 마지막 확인에만 씁니다.

**[직접 작성]** 2-4의 `pipe_model`을 **학습용 데이터로** 5겹 교차검증하세요. 점수 다섯 개와 평균, 표준편차를 출력합니다.

<details><summary>힌트</summary>

```python
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe_model, train_df[features], y_train, cv=cv, scoring="roc_auc")
```
</details>

In [13]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe_model, train_df[features], y_train, cv=cv, scoring="roc_auc")
print("fold별 점수:", scores.round(3))
print(f"평균 {scores.mean():.3f} | 표준편차 {scores.std():.3f}")

fold별 점수: [0.765 0.765 0.759 0.769 0.762]
평균 0.764 | 표준편차 0.003


> **[판단]** 교차검증 평균은 평가용 ROC AUC(0.773)와 얼마나 가깝나요? 교차검증에는 평가용 데이터가 쓰였나요? 파이프라인을 통째로 넣었다는 점은 왜 중요할까요?
>
> 나의 답:

교차검증 평균은 **0.764**(±0.003)로, 평가용 ROC AUC **0.773**보다 0.009 낮습니다. 방향이 맞는 차이입니다. 교차검증은 학습용 36,168명만 썼고(평가용 9,043명은 한 번도 쓰지 않았습니다), fold마다 80%로 학습하니 데이터가 조금 적어 점수가 약간 낮게 나옵니다. 평가용 점수(0.773)는 **실력이 아니라 한 번의 시험 결과**이고, 교차검증 평균이 더 안정적인 추정입니다. 파이프라인을 통째로 넣은 것이 중요한 이유는, 전처리까지 fold 안에서 다시 fit되어야 스케일러·인코더가 검증 fold의 정보를 보지 않기 때문입니다. 전처리를 밖에서 fit한 뒤 교차검증하면 그 자체가 누수입니다.


## 3-2. 두 모델을 비교할 때: 먼저 예상하기

결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다. 다음 절에서 세 쌍의 모델을 같은 5겹 교차검증으로 비교합니다.

| 비교 | 모델 A | 모델 B |
| --- | --- | --- |
| ① | 로지스틱 회귀 ($C = 1$, 기본값) | 로지스틱 회귀 ($C = 0.01$, 규제를 강하게) |
| ② | 결정트리 깊이 5 | 결정트리 깊이 8 |
| ③ | 로지스틱 회귀 ($C = 1$) | 결정트리 깊이 8 |

> **[판단]** 예상: 세 비교 중 "B가 A보다 낫다(또는 못하다)"고 자신 있게 보고할 수 있는 비교는 어느 것일까요? 이유도 적어 보세요.
>
> 나의 답:

예상: **③(로지스틱 vs 트리)** 은 차이가 크게 나서 자신 있게 보고할 수 있고, **①(C=1 vs C=0.01)** 은 차이가 작아 방향은 일관되지만 "낫다"고 보고하기 어려울 것 같습니다. **②(깊이 5 vs 8)** 는 어느 쪽이 나은지 애매할 것으로 봤습니다.


## 3-3. fold별 차이로 비교하기

**[제공]** 모델을 쉽게 만드는 함수 `make_pipe`와, 두 모델을 **같은 5겹**으로 평가해 fold별 점수와 차이를 보여 주는 함수 `compare`입니다.

In [14]:
# [제공] 모델 만들기와 대응 비교
def make_pipe(clf, scale=True):
    pre = ColumnTransformer([
        ("num", StandardScaler() if scale else "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])
    return Pipeline([("pre", pre), ("clf", clf)])

def compare(name_a, model_a, name_b, model_b):
    s_a = cross_val_score(model_a, train_df[features], y_train, cv=cv, scoring="roc_auc")
    s_b = cross_val_score(model_b, train_df[features], y_train, cv=cv, scoring="roc_auc")
    diff = s_b - s_a
    print(f"{name_a}: {s_a.round(3)} | 평균 {s_a.mean():.3f} ± {s_a.std():.3f}")
    print(f"{name_b}: {s_b.round(3)} | 평균 {s_b.mean():.3f} ± {s_b.std():.3f}")
    print(f"fold별 차이 (B - A): {diff.round(3)} | 평균 차이 {diff.mean():+.3f}")
    print()

**[직접 작성]** `make_pipe`로 모델 네 개를 만들고, `compare`로 3-2의 세 쌍을 비교하세요. 결정트리는 스케일링이 필요 없으므로 `scale=False`로 만듭니다(교안 2-6). 트리는 `random_state=42`를 줍니다.

<details><summary>힌트</summary>

```python
logit_1 = make_pipe(LogisticRegression(max_iter=3000))
logit_001 = make_pipe(LogisticRegression(max_iter=3000, C=0.01))
tree_5 = make_pipe(DecisionTreeClassifier(max_depth=5, random_state=42), scale=False)
compare("C=1", logit_1, "C=0.01", logit_001)
```
</details>

In [15]:
logit_1 = make_pipe(LogisticRegression(max_iter=3000))
logit_001 = make_pipe(LogisticRegression(max_iter=3000, C=0.01))
tree_5 = make_pipe(DecisionTreeClassifier(max_depth=5, random_state=42), scale=False)
tree_8 = make_pipe(DecisionTreeClassifier(max_depth=8, random_state=42), scale=False)

compare("① 로지스틱 C=1", logit_1, "① 로지스틱 C=0.01", logit_001)
compare("② 트리 깊이 5", tree_5, "② 트리 깊이 8", tree_8)
compare("③ 로지스틱 C=1", logit_1, "③ 트리 깊이 5", tree_5)

① 로지스틱 C=1: [0.765 0.765 0.759 0.769 0.762] | 평균 0.764 ± 0.003
① 로지스틱 C=0.01: [0.763 0.76  0.759 0.765 0.761] | 평균 0.761 ± 0.002
fold별 차이 (B - A): [-0.002 -0.005 -0.    -0.004 -0.001] | 평균 차이 -0.002



② 트리 깊이 5: [0.688 0.713 0.7   0.691 0.699] | 평균 0.698 ± 0.009
② 트리 깊이 8: [0.705 0.715 0.711 0.693 0.711] | 평균 0.707 ± 0.008
fold별 차이 (B - A): [0.016 0.002 0.011 0.002 0.012] | 평균 차이 +0.009



③ 로지스틱 C=1: [0.765 0.765 0.759 0.769 0.762] | 평균 0.764 ± 0.003
③ 트리 깊이 5: [0.688 0.713 0.7   0.691 0.699] | 평균 0.698 ± 0.009
fold별 차이 (B - A): [-0.076 -0.052 -0.06  -0.078 -0.063] | 평균 차이 -0.066



> **[판단]** 비교 ①: fold별 차이의 방향은 일관되나요? 크기는 실무적으로 의미가 있나요? 둘 중 무엇을 고르겠습니까?
>
> 나의 답:

비교 ①은 fold별 차이(-0.002, -0.005, -0.000, -0.004, -0.001)의 **방향이 다섯 번 모두 같습니다**(C=0.01이 조금 낮음). 크기는 평균 -0.002로, 실무적으로 의미가 없습니다. 두 모델의 실력 차이가 아니라 규제가 조금 더 강해 생긴 미세한 차이입니다. 나는 **C=1**을 쓰겠습니다. 더 단순한 모델을 선호할 이유가 없다면 굳이 0.002를 포기할 필요가 없고, 어느 쪽을 써도 결과는 같습니다.


> **[판단]** 비교 ②: 다섯 번 모두 같은 쪽이 이겼나요? 차이의 크기는 fold마다 어떻게 다른가요? "깊이 8이 낫다"고 보고하겠습니까?
>
> 나의 답:

비교 ②는 다섯 번 모두 깊이 8이 이겼습니다(+0.016, +0.002, +0.011, +0.002, +0.012). 방향은 일관되지만 크기는 평균 +0.009이고 fold에 따라 0.002~0.016으로 흔들립니다. 표준편차(0.008~0.009)와 비슷한 크기라, "깊이 8이 확실히 낫다"고 보고하기는 어렵습니다. 저는 "깊이 8이 근소하게 앞서지만 차이가 fold 편차 수준"이라고 적고, 해석이 쉬운 쪽(또는 두 모델을 함께 보고)을 택하겠습니다.


> **[판단]** 비교 ③: 차이가 표준편차의 몇 배쯤 되나요? 이 비교는 "낫다"고 보고할 수 있나요? 3-2의 예상은 맞았나요?
>
> 나의 답:

비교 ③은 차이가 평균 **-0.066**으로, 표준편차(0.003~0.009)의 **7~20배**입니다. 다섯 번 모두 같은 방향(-0.05~-0.08)이므로 "로지스틱 회귀가 트리보다 낫다"고 보고할 수 있습니다. 3-2의 예상이 맞았습니다. 이 데이터는 범주형 변수가 많고 관계가 대체로 단조로워(예: 계약 기간이 길수록 가입률이 높음) 선형 모델이 잘 맞고, 깊이 5 트리는 그 구조를 충분히 담지 못합니다.


## 3-4. 교차검증으로 설정 고르기

**[직접 작성]** `GridSearchCV`로 로지스틱 회귀의 `C`를 0.01, 0.1, 1, 10 중에서 고르세요. 가장 좋은 설정(`best_params_`)과, 후보별 교차검증 평균·표준편차 표를 출력합니다.

<details><summary>힌트</summary>

파이프라인 안의 설정은 `단계이름__설정이름`으로 가리킵니다.
```python
search = GridSearchCV(make_pipe(LogisticRegression(max_iter=3000)), {"clf__C": [0.01, 0.1, 1, 10]},
                      cv=cv, scoring="roc_auc")
search.fit(train_df[features], y_train)
pd.DataFrame(search.cv_results_)[["param_clf__C", "mean_test_score", "std_test_score"]]
```
</details>

In [16]:
search = GridSearchCV(
    make_pipe(LogisticRegression(max_iter=3000)),
    {"clf__C": [0.01, 0.1, 1, 10]},
    cv=cv, scoring="roc_auc",
)
search.fit(train_df[features], y_train)
print("best_params_:", search.best_params_, "| best_score_:", round(search.best_score_, 3))
grid_table = pd.DataFrame(search.cv_results_)[["param_clf__C", "mean_test_score", "std_test_score"]].round(4)
grid_table

best_params_: {'clf__C': 0.1} | best_score_: 0.764


,param_clf__C,mean_test_score,std_test_score
0,0.01,0.7615,0.0021
1,0.10,0.7641,0.0029
2,1.00,0.7639,0.0031
3,10.00,0.7638,0.0031


> **[판단]** `GridSearchCV`가 고른 설정과 다른 후보들의 차이는 3-3의 기준으로 구별할 만한 크기인가요? 여러분이라면 어떤 `C`를 쓰겠습니까?
>
> 나의 답:

GridSearchCV는 **C=0.1**을 골랐습니다(0.7641). 그런데 다른 후보들도 C=1이 0.7639, C=10이 0.7638, C=0.01이 0.7615로 차이가 **0.0002~0.0026**에 불과합니다. 3-3의 기준(차이가 fold 편차보다 큰가)으로 보면 C=0.1, 1, 10은 **구별할 수 없는 크기**이고, C=0.01만 조금 낮습니다. 여러분이라면 C=0.1을 그대로 쓰거나, 해석이 쉬운 C=1을 쓰겠습니다. 저는 "C=0.1이 뽑혔지만 0.1~10은 사실상 같은 모델"이라고 적고 **C=1**을 쓰겠습니다. 규제가 약한 쪽이 계수를 해석하기 쉽고, 차이는 무의미하기 때문입니다.


## 3-5. 시간이 있는 데이터는 시간 순으로

이 파일의 행은 2008년 5월부터 2010년 11월까지 **날짜순**으로 정렬되어 있습니다. **[제공]** 셀은 행을 순서대로 열 묶음으로 나눠 가입률을 보여 줍니다.

In [17]:
# [제공] 시간 순서대로 본 가입률
deciles = pd.Series(np.arange(len(df)) * 10 // len(df), index=df.index)
print(df.groupby(deciles)["target"].mean().round(3).rename("가입률").rename_axis("구간 (0 = 처음 10%)"))

구간 (0 = 처음 10%)
0    0.030
1    0.037
2    0.048
3    0.064
4    0.061
5    0.062
6    0.105
7    0.131
8    0.161
9    0.471
Name: 가입률, dtype: float64


**[직접 작성]** 앞 80%의 행으로 학습하고 뒤 20%의 행으로 평가하는 **시간 순 분할**을 만드세요. 2-4와 같은 파이프라인을 학습용(앞 80%)으로 `fit`하고, 평가용(뒤 20%)의 확률을 `prob_time`에 저장한 뒤 `record("Chapter 3 모델", y_test_time, prob_time, split="시간 순")`으로 기록하세요.

<details><summary>힌트</summary>

```python
cut = int(len(df) * 0.8)
train_time, test_time = df.iloc[:cut], df.iloc[cut:]
y_test_time = test_time["target"]
```
`make_pipe(LogisticRegression(max_iter=3000))`로 새 파이프라인을 만들어 `train_time[features]`로 `fit`합니다.
</details>

In [18]:
cut = int(len(df) * 0.8)
train_time, test_time = df.iloc[:cut], df.iloc[cut:]
y_test_time = test_time["target"]
print("시간 순 분할:", len(train_time), "행 학습 /", len(test_time), "행 평가 | 평가 가입률", round(y_test_time.mean(), 3))

pipe_time = make_pipe(LogisticRegression(max_iter=3000))
pipe_time.fit(train_time[features], train_time["target"])
prob_time = pipe_time.predict_proba(test_time[features])[:, 1]
record("Chapter 3 모델", y_test_time, prob_time, split="시간 순")

시간 순 분할: 36168 행 학습 / 9043 행 평가 | 평가 가입률 0.316
기록: [Chapter 3 모델 / 시간 순] ROC AUC 0.692 | 상위 1,000명 가입 532명 (무작위의 1.7배)


> **[판단]** 시간 순으로 나누면 ROC AUC가 얼마나 달라지나요? 왜 이런 차이가 생길까요? 실제 캠페인과 더 가까운 조건은 어느 쪽인가요?
>
> 나의 답:

시간 순으로 나누면 ROC AUC가 **0.773 → 0.692**로 **0.081** 떨어집니다. 이유는 학습한 기간(2008.5~2010.5 무렵)과 평가한 기간(2010.6~2010.11)의 **고객·시장 조건이 다르기** 때문입니다. 시간 순 분할의 평가용 가입률은 31.6%로, 무작위 분할의 11.7%보다 훨씬 높습니다(마지막 구간의 가입률이 47.1%). 모델은 가입률이 11.7%이던 시절의 패턴을 배웠는데, 평가 구간은 분위기가 달라졌습니다. 실제 캠페인과 더 가까운 조건은 **시간 순 분할**입니다. 캠페인은 항상 과거 데이터로 배워 미래에 적용하기 때문입니다.


> **[판단]** 그런데 시간 순 분할의 "상위 1,000명 가입"은 무작위 분할보다 많거나 적은가요? ROC AUC는 낮아졌는데 이 숫자는 왜 이렇게 나왔을까요? 결과표의 "평가 가입률"과 "배수" 열을 함께 보세요.
>
> 나의 답:

시간 순 분할의 "상위 1,000명 가입"은 **532명**으로 무작위 분할(464명)보다 많습니다. ROC AUC는 낮아졌는데(0.692) 이 숫자가 큰 이유는 **평가 가입률이 31.6%로 높기** 때문입니다. 무작위로 1,000명을 골라도 316명이 가입하는 구간이라, 모델이 532명을 얻은 것은 1.7배에 그칩니다(무작위 분할은 4.0배). 즉 "가입 수"는 데이터의 가입률에 따라 달라지므로, 분할 방식이 다른 두 숫자를 그대로 비교하면 안 됩니다. 결과표의 '배수' 열이 이 차이를 정확히 보여 줍니다.


---
# Part 4. 보고할 숫자 정하기

이제 질문으로 돌아옵니다. **마케팅팀에 "다음 캠페인에서 기대할 가입 수"로 보고할 숫자는 어느 것인가?**

## 4-1. 먼저 예상하기

결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다. 다음 절에서 네 가지 조합의 점수를 한 표에 놓습니다.

| | 무작위 분할 | 시간 순 분할 |
| --- | --- | --- |
| 통화 시간 없음 | 1-2 (기록됨) | 3-5 (기록됨) |
| 통화 시간 있음 | 1-3 (기록됨) | 4-2에서 만듦 |

> **[판단]** 예상: 네 조합 중 ROC AUC가 가장 높은 것과 가장 낮은 것은 무엇일까요? 다음 캠페인에서 실제로 나올 성과에 가장 가까운 것은 어느 칸일까요?
>
> 나의 답:

예상: ROC AUC는 **통화 시간 + 무작위(0.906)** 가 가장 높고, **통화 시간 없음 + 시간 순(0.692)** 이 가장 낮을 것으로 봤습니다. 다음 캠페인 성과에 가장 가까운 칸은 **통화 시간 없음 + 시간 순**입니다. 통화 시간이 들어간 칸은 애초에 쓸 수 없는 모델이고, 무작위 분할은 미래를 섞어 평가한 낙관적 조건이기 때문입니다.


## 4-2. 네 조합 비교하기

**[직접 작성]** 남은 한 칸(통화 시간 있음 + 시간 순 분할)을 만드세요. `features + ["duration"]`을 쓰는 파이프라인을 만들어야 하므로, `make_pipe`의 숫자 열 목록에 `duration`을 더해야 합니다. 확률을 기록하고 `show_log()`로 결과표 전체를 보세요.

<details><summary>힌트</summary>

`make_pipe`는 `num_cols`를 씁니다. 이 셀에서만 직접 만들면 됩니다.
```python
pre_dur = ColumnTransformer([
    ("num", StandardScaler(), num_cols + ["duration"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
```
</details>

In [19]:
pre_dur = ColumnTransformer([
    ("num", StandardScaler(), num_cols + ["duration"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
pipe_dur_time = Pipeline([("pre", pre_dur), ("clf", LogisticRegression(max_iter=3000))])
pipe_dur_time.fit(train_time[features + ["duration"]], train_time["target"])
prob_dur_time = pipe_dur_time.predict_proba(test_time[features + ["duration"]])[:, 1]
record("통화 시간 추가", y_test_time, prob_dur_time, split="시간 순")

show_log()

기록: [통화 시간 추가 / 시간 순] ROC AUC 0.782 | 상위 1,000명 가입 598명 (무작위의 1.9배)


,실험,분할,ROC AUC,PR AUC,평가 가입률,"상위 1,000명 가입","무작위 1,000명 기대",배수
0,Chapter 3 모델,무작위,0.773,0.411,0.117,464,117,4.0
1,통화 시간 추가,무작위,0.906,0.544,0.117,578,117,4.9
2,파이프라인,무작위,0.773,0.411,0.117,465,117,4.0
3,Chapter 3 모델,시간 순,0.692,0.496,0.316,532,316,1.7
4,통화 시간 추가,시간 순,0.782,0.569,0.316,598,316,1.9


> **[판단]** 예상은 맞았나요? 네 조합 중 마케팅팀에 보고할 점수는 어느 칸이고, 왜 그런가요? 통화 시간을 넣은 두 칸은 왜 보고할 수 없나요?
>
> 나의 답:

예상은 맞았습니다. 네 칸의 ROC AUC는 0.906(시간 포함·무작위) > 0.782(시간 포함·시간 순) > 0.773(시간 없음·무작위) > 0.692(시간 없음·시간 순)입니다. 마케팅팀에 보고할 숫자는 **통화 시간 없음 + 시간 순 분할**의 **0.692**입니다. 이 칸만 "예측 시점에 아는 변수"로 "미래 기간"을 평가했기 때문입니다. 통화 시간을 넣은 두 칸(0.906, 0.782)은 전화가 끝나야 알 수 있는 값이라 다음 캠페인에서 **전화하기 전에 쓸 수 없습니다**. 무작위 분할 칸(0.773)도 미래 고객이 학습에 섞여 낙관적입니다. Chapter 3에서 0.773을 보고했다면, 정직한 숫자는 0.692였던 것입니다.


> **[판단]** 마케팅팀은 "1,000통이면 몇 명이 가입하나요?"라고 묻습니다. 시간 순 분할의 "상위 1,000명 가입" 숫자를 그대로 약속해도 될까요? 다음 캠페인의 가입률이 지난 기간과 다를 수 있다는 점을 생각하면, 어떤 형태로 말하는 것이 정직할까요?
>
> 나의 답:

시간 순 분할의 "상위 1,000명 가입 532명"을 그대로 약속하면 안 됩니다. 이 숫자는 평가 구간의 가입률 31.6%에서 나온 것이고, 다음 캠페인의 가입률이 11.7%라면 같은 모델로도 1,000통에서 150~200명 수준에 그칠 수 있습니다. 정직한 표현은 **조건을 붙인 범위**입니다. "지난 기간의 마지막 3개월처럼 가입률이 30% 안팎이라면 1,000통에서 500명 전후, 가입률이 12% 수준이라면 1,000통에서 180명 전후를 기대할 수 있다. 어느 쪽이든 무작위의 1.7~4배"처럼 **가입률에 따라 달라진다**는 것을 함께 말하고, 캠페인 초반 100통의 결과로 가입률을 다시 추정해 예상치를 조정하는 방법을 제안하겠습니다.


## 4-3. 점검표 채우기

교안 3-6의 여섯 질문을 **보고하려는 모델**(통화 시간 없음, 파이프라인)에 대해 하나씩 답하세요. 각 질문에 "통과 / 미통과 / 확인 못 함"과 근거를 적습니다.

> **[판단]** 점검표:
>
> ① 쓴 변수가 모두 예측 시점에 존재하는가:
>
> ② 전처리 기준을 학습용으로만 잡았는가:
>
> ③ 설정을 고를 때 평가용 데이터를 쓰지 않았는가:
>
> ④ 비교한 차이가 fold 편차보다 크고 의미 있는가:
>
> ⑤ 평가용이 학습용보다 나중인가:
>
> ⑥ 베이스라인과 현행 규칙보다 얼마나 나은가:
>
> 나의 답:

① 쓴 변수가 모두 예측 시점에 존재하는가: **부분적으로 O**. `duration`은 뺐지만 `campaign`(마지막 통화 포함)과 `month`·`day`(마지막 통화의 월·일)는 여전히 미묘합니다. 캠페인 시작 시점 값으로 고정한다는 전제가 필요합니다.
② 전처리 기준을 학습용으로만 잡았는가: **O**. 2-4의 파이프라인이 학습용 안에서만 fit됩니다.
③ 설정을 고를 때 평가용 데이터를 쓰지 않았는가: **O**. 교차검증으로 골랐고, 평가용은 마지막 확인에만 썼습니다.
④ 비교한 차이가 fold 편차보다 크고 의미 있는가: **부분적으로 O**. 로지스틱 vs 트리(-0.066)는 크고, C 조정(-0.002)·깊이 5 vs 8(+0.009)은 편차 수준이라 판단을 보류했습니다.
⑤ 평가용이 학습용보다 나중인가: **O**. 시간 순 분할로 확인했고, 점수는 0.692로 낮아졌습니다.
⑥ 베이스라인과 현행 규칙보다 얼마나 나은가: **O**. 베이스라인 정확도 0.883 대비 재현율이 의미 있고, Chapter 3에서 현행 규칙보다 상위 인원에서 1.06~1.13배 우수했습니다.
결론: ①과 ④가 완전히 해결되지 않았습니다. 이 상태로 "0.692"를 보고하되, 캠페인 시작 시점 기준으로 변수를 고정하는 작업이 남아 있다고 적어야 합니다.


## 4-4. 마케팅팀 보고 메모

분석 결과를 마케팅팀장에게 메시지로 보낸다고 생각하고 쓰세요. 팀장은 머신러닝 용어를 모릅니다. 아래 네 가지가 모두 들어가야 합니다.

1. **보고하는 숫자와 그 근거**: 어떤 조건(분할 방식)에서 잰 숫자인지
2. **통화 시간 모델을 쓰지 않는 이유**: 점수가 더 높은데도 쓰지 않는 이유를 쉬운 말로
3. **1,000통의 기대 효과**: 절대 숫자를 약속할 수 있는지, 아니라면 어떤 형태로 말하는지
4. **주의할 점**: 다음 캠페인의 가입률이 달라질 가능성, 아직 확인하지 못한 것

> **[판단]** 마케팅팀 보고 메모:
>
> 1. 보고하는 숫자와 근거:
>
> 2. 통화 시간 모델을 쓰지 않는 이유:
>
> 3. 1,000통의 기대 효과:
>
> 4. 주의할 점:
>
> 나의 답:

1. **보고하는 숫자와 근거**: 다음 캠페인에서 기대할 성과는 **ROC AUC 0.692**, 1,000통에서 **무작위의 1.7배**입니다. 이 숫자는 통화 시간을 뺀 12개 변수로, **시간 순 분할**(앞 80% 학습·뒤 20% 평가)에서 측정한 것입니다. 무작위 분할의 0.773은 미래 고객이 학습에 섞인 낙관적 값이라 보고하지 않습니다.
2. **통화 시간 모델을 쓰지 않는 이유**: AUC가 0.906까지 오르지만, `duration`은 전화가 끝나야 생기는 값입니다. 전화할 고객을 고르는 시점에는 존재하지 않으므로, 캠페인에서 쓸 수 없는 모델입니다. 60초 미만 통화의 가입률이 0.2%, 600초 초과가 48.6%라는 것도 "가입할 고객이라서 통화가 길어진" 결과로 읽는 것이 맞습니다.
3. **1,000통의 기대 효과**: 시간 순 조건에서 1,000통으로 무작위의 1.7배(532명, 평가 가입률 31.6% 구간 기준)입니다. 가입률이 12% 수준인 구간이라면 1,000통에서 180~200명 선으로 봐야 합니다. 정확한 약속 대신 **가입률에 따른 범위**로 제시합니다.
4. **주의할 점**: ① `campaign`·`month`·`day`는 "마지막 통화" 기준이라 캠페인 시작 시점 값으로 고정해야 합니다. ② 시간 순 분할에서 AUC가 0.08 떨어졌으므로, 시장이 바뀌면 성능이 더 떨어질 수 있습니다. ③ 이 데이터는 2008~2010년 포르투갈 은행의 것이고, 가입률이 기간에 따라 4%에서 47%까지 달라졌습니다. ④ 캠페인 초반 100~200통의 실제 결과로 예상치를 다시 조정할 것을 권합니다.


---
# 마무리

## 이 Chapter의 판단 포인트

> **이 정보는 예측 시점에 알 수 있는가? 그리고 두 모델의 점수 차이는 교차검증 편차보다 큰가?**

## 개인 검증 체크리스트에 추가

오늘 실습에서 내가 실제로 겪은 일을 바탕으로, 체크리스트의 줄을 내 말로 적으세요.

예: "점수가 갑자기 좋아진 변수를 보면, 그 값이 언제 기록되는지부터 확인한다."

> **[판단]** 나의 체크리스트 6번(누수를 점검할 때):
>
> 나의 답:

누수를 점검할 때는 (1) 각 변수가 **예측 시점에 실제로 존재하는지**를 변수 설명서가 아니라 "업무 절차상 언제 채워지는가"로 확인하고, (2) 전처리(스케일러·인코더·타깃 인코딩)가 **학습용 안에서만** fit되는지(파이프라인), (3) 평가지표가 갑자기 크게 오르면 그 상승이 어떤 변수에서 왔는지를 먼저 의심한다. AUC가 한 변수로 0.13 오르면 성능이 아니라 누수다.


> **[판단]** 나의 체크리스트 7번(점수를 비교하고 보고할 때):
>
> 나의 답:

점수를 비교하고 보고할 때는 (1) 비교하는 점수가 **같은 분할 방식**인지 확인하고(무작위 vs 시간 순을 섞지 않기), (2) 차이가 **fold 편차보다 큰지** 보고, (3) 보고할 숫자는 "예측 시점에 아는 변수 + 미래 기간"으로 측정한 **가장 낮은 정직한 숫자**로 고르고, (4) 그 숫자가 어떤 조건(가입률, 기간)에서 나왔는지를 함께 적는다.


## 제출물

- 이 노트북 (모든 **[판단]** 칸을 채운 상태)

수고했습니다. 다음 Chapter 5에서는 트리를 여러 개 모아 더 강한 모델을 만들고, 그 모델이 무엇을 보고 예측하는지 SHAP으로 설명합니다.